# Gen AI Fundamentals: Teaching LLMs to Reason with GRPO

## Project: Reasoning Engine — Letter Counting via RL Fine-Tuning

**Goal:** Fine-tune `Qwen2.5-3B-Instruct` to reliably count letter occurrences in a word using step-by-step reasoning, trained via GRPO reinforcement learning + LoRA.

**Key Technologies:** LoRA · Unsloth · vLLM · GRPO (TRL)

## Phase 1: Project Setup

In [ ]:
# Cell 1 — Install dependencies (pre-installed in Vocareum; uncomment for Colab/local)
# !pip install unsloth vllm trl peft accelerate datasets transformers bitsandbytes -q

In [ ]:
# Cell 2 — Verify GPU
!nvidia-smi

In [ ]:
# Cell 3 — Imports
import re, random, torch, numpy as np, pandas as pd, matplotlib.pyplot as plt
from datasets import Dataset
from trl import GRPOTrainer, GRPOConfig
from unsloth import FastLanguageModel

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
# Cell 4 — Constants
MODEL_NAME     = "Qwen/Qwen2.5-3B-Instruct"
MAX_SEQ_LENGTH = 1024
DTYPE          = None        # auto-detect (bfloat16 on Ampere+, float16 on T4)
LOAD_IN_4BIT   = True        # 4-bit quant to fit 3B model in 16 GB VRAM

In [ ]:
# Cell 5 — TODO: Load model + apply LoRA
#
# lora_rank = 64
#   Controls the dimensionality of the low-rank adapter matrices (A and B).
#   rank=64 gives enough expressiveness to learn a complex procedural pattern
#   (step-by-step letter counting) while fitting comfortably in 16 GB VRAM.
#   rank=8/16 would likely underfit; rank=128+ wastes memory with little gain.
#
# target_modules — ALL attention + MLP linear projections:
#   q_proj, k_proj, v_proj, o_proj  →  teach the model WHAT to attend to
#   gate_proj, up_proj, down_proj   →  teach the model HOW to process it
#   Targeting all 7 layers gives maximum LoRA expressiveness while still
#   being far cheaper than full fine-tuning.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = MODEL_NAME,
    max_seq_length= MAX_SEQ_LENGTH,
    dtype         = DTYPE,
    load_in_4bit  = LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r               = 64,                   # lora_rank
    target_modules  = [
        "q_proj", "k_proj", "v_proj", "o_proj",     # attention projections
        "gate_proj", "up_proj", "down_proj",          # MLP / feed-forward
    ],
    lora_alpha              = 64,           # scaling factor (= rank is standard)
    lora_dropout            = 0,            # 0 for Unsloth-optimised kernels
    bias                    = "none",
    use_gradient_checkpointing = "unsloth", # saves VRAM during backprop
    random_state            = 42,
    use_rslora              = False,
    loftq_config            = None,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params : {trainable:,}  ({100*trainable/total:.2f}% of {total:,})")

## Phase 2: Prompt Engineering Baseline

In [ ]:
# Cell 6 — Baseline: blank system prompt (poor performance expected)
BLANK_SYSTEM_PROMPT = ""

messages = [
    {"role": "system", "content": BLANK_SYSTEM_PROMPT},
    {"role": "user",   "content": "How many times does the letter 'e' appear in the word 'effectiveness'?"},
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)

FastLanguageModel.for_inference(model)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=200, temperature=0.7,
                         do_sample=True, pad_token_id=tokenizer.eos_token_id)
response = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("BASELINE (no system prompt):")
print("-" * 50)
print(response)

In [ ]:
# Cell 7 — TODO: Improved SYSTEM_PROMPT with Chain-of-Thought + one-shot example
#
# Design choices:
#   1. Chain-of-Thought: explicit instruction to go letter-by-letter with a running total
#   2. One-shot example (word="room", letter="o") so the model can mirror the exact format
#   3. Strict XML output tags <reasoning> and <answer> so reward functions can parse output

SYSTEM_PROMPT = """\
You are a precise letter-counting assistant.
When asked how many times a letter appears in a word, reason step by step:

1. Write each letter of the word with its position number.
2. State whether that letter matches the target letter.
3. Keep a running total after each letter.
4. Give the final count inside <answer> tags.

Always use this EXACT format:
<reasoning>
[step-by-step work]
</reasoning>
<answer>[digit]</answer>

Example:
Question: How many of the letter "o" are there in the word "room"?
<reasoning>
1. r - is 'r' == 'o'? No.  Running total: 0
2. o - is 'o' == 'o'? Yes! Running total: 1
3. o - is 'o' == 'o'? Yes! Running total: 2
4. m - is 'm' == 'o'? No.  Running total: 2
The letter 'o' appears 2 times in 'room'.
</reasoning>
<answer>2</answer>
"""

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": "How many times does the letter 'e' appear in the word 'effectiveness'?"},
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)

FastLanguageModel.for_inference(model)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=400, temperature=0.7,
                         do_sample=True, pad_token_id=tokenizer.eos_token_id)
response = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("IMPROVED (CoT + one-shot system prompt):")
print("-" * 50)
print(response)

## Phase 3: Dataset Creation

In [ ]:
# Cell 8 — Word list
ALL_WORDS = [
    "apple", "banana", "cherry", "dragon", "elephant",
    "flower", "garden", "jungle", "rainbow", "thunder",
    "freedom", "gravity", "harmony", "island", "kitchen",
    "lantern", "monster", "network", "optical", "pattern",
    "quantum", "shelter", "umbrella", "village", "whisper",
    "yellow", "zebra", "balance", "captain", "diamond",
    "energy", "fantasy", "general", "history", "integer",
    "journey", "kingdom", "library", "mystery", "natural",
    "opinion", "process", "quality", "reality", "science",
    "testing", "uniform", "victory", "warning", "example",
    "program", "student", "teacher", "problem", "answer",
    "matrix", "vector", "factor", "method", "system",
    "ribbon", "silver", "temple", "winter", "zipper",
    "mirror", "pillow", "hammer", "amazing", "xylophone",
]
print(f"Total words: {len(ALL_WORDS)}")

In [ ]:
# Cell 9 — generate_records function
random.seed(42)

def generate_records(words, system_prompt):
    records = []
    for word in words:
        for letter in set(word.lower()):
            count = word.lower().count(letter)
            records.append({
                "prompt": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user",   "content":
                        f"How many times does the letter '{letter}' appear in the word '{word}'?"},
                ],
                "word":   word,
                "letter": letter,
                "answer": str(count),
            })
    random.shuffle(records)
    return records

records = generate_records(ALL_WORDS, SYSTEM_PROMPT)
ds = Dataset.from_list(records)
print(f"Dataset size: {len(ds)} records")
print(f"Sample — word: {ds[0]['word']}, letter: {ds[0]['letter']}, answer: {ds[0]['answer']}")

In [ ]:
# Cell 10 — Untuned model on a dataset sample (baseline comparison)
sample     = ds[0]
user_msg   = sample["prompt"][-1]["content"]
gt_answer  = sample["answer"]

messages = sample["prompt"]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)

FastLanguageModel.for_inference(model)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=400, temperature=0.7,
                         do_sample=True, pad_token_id=tokenizer.eos_token_id)
response = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print(f"Question : {user_msg}")
print(f"GT answer: {gt_answer}")
print("-" * 50)
print(response)

## Phase 4: Building the Reward Functions

Five reward functions covering **format, numbering, spelling, counting, and correctness**.

In [ ]:
# Cell 11 — numbering_reward_func
# Rewards in-order numbering (+0.5), penalises out-of-order (-0.5),
# penalises steps beyond the word length (-1.0).

def numbering_reward_func(prompts, completions, word, **kwargs):
    rewards = []
    for completion, w in zip(completions, word):
        text = completion[0]["content"] if isinstance(completion, list) else completion
        reward = 0.0
        reasoning_match = re.search(r"<reasoning>(.*?)</reasoning>", text, re.DOTALL)
        if reasoning_match:
            reasoning_text = reasoning_match.group(1)
            numbers_found  = [int(n) for n in re.findall(r"^\s*(\d+)[.)]", reasoning_text, re.MULTILINE)]
            expected = 1
            for num in numbers_found:
                if num > len(w):
                    reward -= 1.0   # hallucinated extra step
                elif num == expected:
                    reward += 0.5   # correct in-order step
                    expected += 1
                else:
                    reward -= 0.5   # out-of-order
        rewards.append(reward)
    return rewards

# ── Validation ──────────────────────────────────────────────────────────────
correct_response = [
    [{"content": """<reasoning>
1. r - No. Running total: 0
2. o - Yes! Running total: 1
3. o - Yes! Running total: 2
4. m - No. Running total: 2
</reasoning><answer>2</answer>"""}]
]
wrong_response = [
    [{"content": """<reasoning>
1. r - No. 0
3. o - Yes! 1
2. o - Yes! 2
5. m - No. 2
6. x - extra
</reasoning><answer>1</answer>"""}]
]

r_correct = numbering_reward_func(None, correct_response, ["room"])
r_wrong   = numbering_reward_func(None, wrong_response,   ["room"])
print(f"Correct numbering reward : {r_correct[0]:.2f}  (should be > 0)")
print(f"Wrong   numbering reward : {r_wrong[0]:.2f}  (should be < correct)")
assert r_correct[0] > r_wrong[0], "numbering_reward_func failed!"
print("PASSED")

In [ ]:
# Cell 12 — spelling_reward_func
# +2.0 exact spelling | -1.0/extra letter | -0.5/missing letter | -0.5/mismatch

def spelling_reward_func(prompts, completions, word, **kwargs):
    rewards = []
    for completion, w in zip(completions, word):
        text    = completion[0]["content"] if isinstance(completion, list) else completion
        w_lower = list(w.lower())
        reasoning_match = re.search(r"<reasoning>(.*?)</reasoning>", text, re.DOTALL)
        if not reasoning_match:
            rewards.append(-1.0)
            continue
        extracted = [l.lower() for l in
                     re.findall(r"^\s*\d+[.)]\s+([a-zA-Z])", reasoning_match.group(1), re.MULTILINE)]
        if extracted == w_lower:
            rewards.append(2.0)
            continue
        reward  = 0.0
        diff    = len(extracted) - len(w_lower)
        reward += (-1.0 * diff)    if diff > 0 else (0.5 * diff)   # extra / missing
        overlap = min(len(extracted), len(w_lower))
        reward -= 0.5 * sum(a != b for a, b in zip(extracted[:overlap], w_lower[:overlap]))
        rewards.append(reward)
    return rewards

# ── Validation ──────────────────────────────────────────────────────────────
correct_spell = [[{"content": """<reasoning>
1. r - No. 0
2. o - Yes! 1
3. o - Yes! 2
4. m - No. 2
</reasoning><answer>2</answer>"""}]]

wrong_spell = [[{"content": """<reasoning>
1. x - No. 0
2. y - No. 0
3. z - No. 0
4. m - No. 0
5. n - extra
</reasoning><answer>0</answer>"""}]]

rs_correct = spelling_reward_func(None, correct_spell, ["room"])
rs_wrong   = spelling_reward_func(None, wrong_spell,   ["room"])
print(f"Correct spelling reward : {rs_correct[0]:.2f}  (should be +2.0)")
print(f"Wrong   spelling reward : {rs_wrong[0]:.2f}  (should be < 0)")
assert rs_correct[0] > rs_wrong[0], "spelling_reward_func failed!"
print("PASSED")

In [ ]:
# Cell 13 — counting_reward_func
# +1.0 accurate running total at each step | -1.0 inaccurate
# Final reward = mean over all steps → range [-1, +1]

def counting_reward_func(prompts, completions, word, letter, **kwargs):
    rewards = []
    for completion, w, l in zip(completions, word, letter):
        text    = completion[0]["content"] if isinstance(completion, list) else completion
        w_lower = w.lower()
        l_lower = l.lower()
        reasoning_match = re.search(r"<reasoning>(.*?)</reasoning>", text, re.DOTALL)
        if not reasoning_match:
            rewards.append(-1.0)
            continue
        steps = re.findall(
            r"^\s*\d+[.)]\s+([a-zA-Z]).*?(?:[Rr]unning\s+[Tt]otal|total)[:\s]+?(\d+)",
            reasoning_match.group(1), re.MULTILINE)
        if not steps:
            rewards.append(-1.0)
            continue
        step_scores, running = [], 0
        for idx, (step_letter, stated) in enumerate(steps):
            actual = w_lower[idx] if idx < len(w_lower) else step_letter.lower()
            if actual == l_lower:
                running += 1
            step_scores.append(1.0 if int(stated) == running else -1.0)
        rewards.append(sum(step_scores) / len(step_scores))
    return rewards

# ── Validation ──────────────────────────────────────────────────────────────
correct_count = [[{"content": """<reasoning>
1. r - is 'r' == 'o'? No.  Running total: 0
2. o - is 'o' == 'o'? Yes! Running total: 1
3. o - is 'o' == 'o'? Yes! Running total: 2
4. m - is 'm' == 'o'? No.  Running total: 2
</reasoning><answer>2</answer>"""}]]

wrong_count = [[{"content": """<reasoning>
1. r - No.  Running total: 0
2. o - Yes! Running total: 5
3. o - Yes! Running total: 3
4. m - No.  Running total: 1
</reasoning><answer>1</answer>"""}]]

rc_correct = counting_reward_func(None, correct_count, ["room"], ["o"])
rc_wrong   = counting_reward_func(None, wrong_count,   ["room"], ["o"])
print(f"Correct counting reward : {rc_correct[0]:.2f}  (should be +1.0)")
print(f"Wrong   counting reward : {rc_wrong[0]:.2f}  (should be < 0)")
assert rc_correct[0] > rc_wrong[0], "counting_reward_func failed!"
print("PASSED")

In [ ]:
# Cell 14 — format_reward_func
# +0.5 correct <reasoning>...</reasoning><answer>...</answer> format
# +0.5 <answer> tag contains a digit

def format_reward_func(prompts, completions, **kwargs):
    rewards = []
    for completion in completions:
        text   = completion[0]["content"] if isinstance(completion, list) else completion
        reward = 0.0
        if re.search(r"<reasoning>.*?</reasoning>", text, re.DOTALL) and            re.search(r"<answer>.*?</answer>",       text, re.DOTALL):
            reward += 0.5
        if re.search(r"<answer>\s*\d+\s*</answer>", text):
            reward += 0.5
        rewards.append(reward)
    return rewards

# ── Validation ──────────────────────────────────────────────────────────────
correct_fmt = [[{"content": "<reasoning>1. r No. 0</reasoning><answer>2</answer>"}]]
wrong_fmt   = [[{"content": "The answer is two."}]]

rf_correct = format_reward_func(None, correct_fmt)
rf_wrong   = format_reward_func(None, wrong_fmt)
print(f"Correct format reward : {rf_correct[0]:.2f}  (should be +1.0)")
print(f"Wrong   format reward : {rf_wrong[0]:.2f}  (should be 0.0)")
assert rf_correct[0] > rf_wrong[0], "format_reward_func failed!"
print("PASSED")

In [ ]:
# Cell 15 — correct_answer_reward_func
# +2.0 correct final answer | -1.0 incorrect

def correct_answer_reward_func(prompts, completions, answer, **kwargs):
    rewards = []
    for completion, gt in zip(completions, answer):
        text = completion[0]["content"] if isinstance(completion, list) else completion
        m    = re.search(r"<answer>\s*(\d+)\s*</answer>", text)
        pred = m.group(1).strip() if m else (re.search(r"\b(\d+)\b", text) or type("", (), {"group": lambda s,x: "-1"})()).group(1)
        rewards.append(2.0 if pred == str(gt) else -1.0)
    return rewards

# ── Validation ──────────────────────────────────────────────────────────────
correct_ans = [[{"content": "<reasoning>steps</reasoning><answer>2</answer>"}]]
wrong_ans   = [[{"content": "<reasoning>steps</reasoning><answer>9</answer>"}]]

ra_correct = correct_answer_reward_func(None, correct_ans, ["2"])
ra_wrong   = correct_answer_reward_func(None, wrong_ans,   ["2"])
print(f"Correct answer reward : {ra_correct[0]:.2f}  (should be +2.0)")
print(f"Wrong   answer reward : {ra_wrong[0]:.2f}  (should be -1.0)")
assert ra_correct[0] == 2.0  and ra_wrong[0] == -1.0, "correct_answer_reward_func failed!"
print("PASSED")

In [ ]:
# Cell 16 — Combined sanity check across all 5 reward functions
PERFECT = [[{"content": """<reasoning>
1. r - is 'r' == 'o'? No.  Running total: 0
2. o - is 'o' == 'o'? Yes! Running total: 1
3. o - is 'o' == 'o'? Yes! Running total: 2
4. m - is 'm' == 'o'? No.  Running total: 2
The letter 'o' appears 2 times.
</reasoning><answer>2</answer>"""}]]

TERRIBLE = [[{"content": "I think it might be seven?"}]]

tests = [
    ("format",         lambda c,w,l: format_reward_func(None, c)),
    ("numbering",      lambda c,w,l: numbering_reward_func(None, c, w)),
    ("spelling",       lambda c,w,l: spelling_reward_func(None, c, w)),
    ("counting",       lambda c,w,l: counting_reward_func(None, c, w, l)),
    ("correct_answer", lambda c,w,l: correct_answer_reward_func(None, c, ["2"])),
]
print(f"{'Function':<20} {'Perfect':>10} {'Terrible':>10} {'Pass?':>8}")
print("-" * 52)
all_ok = True
for name, fn in tests:
    rp = fn(PERFECT,  ["room"], ["o"])[0]
    rt = fn(TERRIBLE, ["room"], ["o"])[0]
    ok = rp > rt
    all_ok &= ok
    print(f"{name:<20} {rp:>10.2f} {rt:>10.2f} {'✓' if ok else '✗':>8}")
print("\nAll reward functions working!" if all_ok else "\n⚠ Check failed functions!")

## Phase 5: Model Training

In [ ]:
# Cell 17 — TODO: GRPO training hyperparameters
#
# learning_rate = 1e-5
#   Standard LoRA fine-tuning LR. Low enough to avoid catastrophic forgetting,
#   high enough to learn the new procedural skill within 80 steps.
#
# beta = 0.0001
#   KL-divergence penalty. Very small → less constraint on policy drift,
#   acceptable because LoRA inherently limits how far weights can shift.
#
# per_device_train_batch_size = 16
#   16 prompts per GPU step. Fits in 16 GB VRAM with 4-bit quant + LoRA.
#
# num_generations = 4
#   Completions per prompt for the GRPO group comparison.
#   4 gives useful diversity without excessive memory cost.
#
# gradient_accumulation_steps = 1
#   Batch size of 16 already provides adequate gradient signal per step.

COMMON_GRPO_TRAINING_PARAMS = dict(
    learning_rate               = 1e-5,
    beta                        = 0.0001,
    per_device_train_batch_size = 16,
    num_generations             = 4,
    gradient_accumulation_steps = 1,
    adam_beta1                  = 0.9,
    adam_beta2                  = 0.99,
    weight_decay                = 0.1,
    warmup_ratio                = 0.1,
    lr_scheduler_type           = "cosine",
    optim                       = "adamw_8bit",
    logging_steps               = 1,
    bf16                        = torch.cuda.is_bf16_supported(),
    fp16                        = not torch.cuda.is_bf16_supported(),
    output_dir                  = "./grpo_output",
    seed                        = 42,
)

reward_funcs = [
    format_reward_func,
    numbering_reward_func,
    spelling_reward_func,
    counting_reward_func,
    correct_answer_reward_func,
]

print("GRPO params set. reward_funcs ready:", [f.__name__ for f in reward_funcs])

In [ ]:
# Cell 18 — Quick Train (5 steps) — verify reward functions produce non-zero values
print("Quick training run: 5 steps")
print("Watch: all reward columns should show non-zero values.")
print("=" * 60)

FastLanguageModel.for_training(model)

quick_config = GRPOConfig(
    max_steps            = 5,
    max_prompt_length    = 512,
    max_completion_length= 512,
    use_vllm             = True,
    vllm_gpu_memory_utilization = 0.3,
    report_to            = "none",
    **COMMON_GRPO_TRAINING_PARAMS,
)

quick_trainer = GRPOTrainer(
    model            = model,
    processing_class = tokenizer,
    reward_funcs     = reward_funcs,
    args             = quick_config,
    train_dataset    = ds,
)

quick_trainer.train()
print("Quick train complete!")

In [ ]:
# Cell 19 — Log table from quick training
if quick_trainer.state.log_history:
    log_df = pd.DataFrame(quick_trainer.state.log_history)
    print("Quick training log:")
    display(log_df)

In [ ]:
# Cell 20 — Full training run (80 steps, ~30-60 min on T4)
# Expected: 'reward' and 'rewards/correct_answer_reward_func/mean' trend upward.

LONGER_MAX_STEPS = 80

print(f"Full training run: {LONGER_MAX_STEPS} steps (~30-60 min on T4)")
print("=" * 60)

FastLanguageModel.for_training(model)

full_config = GRPOConfig(
    max_steps            = LONGER_MAX_STEPS,
    max_prompt_length    = 512,
    max_completion_length= 512,
    use_vllm             = True,
    vllm_gpu_memory_utilization = 0.3,
    report_to            = "none",
    **COMMON_GRPO_TRAINING_PARAMS,
)

full_trainer = GRPOTrainer(
    model            = model,
    processing_class = tokenizer,
    reward_funcs     = reward_funcs,
    args             = full_config,
    train_dataset    = ds,
)

full_trainer.train()
print("Full training complete!")

In [ ]:
# Cell 21 — Plot training rewards
if full_trainer.state.log_history:
    log_df = pd.DataFrame(full_trainer.state.log_history)
    print("Training log (last 10 steps):")
    display(log_df.tail(10))

    reward_cols = [c for c in log_df.columns if "reward" in c.lower()]
    if reward_cols and "step" in log_df.columns:
        fig, axes = plt.subplots(len(reward_cols), 1,
                                  figsize=(12, 3*len(reward_cols)), squeeze=False)
        for ax, col in zip(axes[:,0], reward_cols):
            valid = log_df[["step", col]].dropna()
            ax.plot(valid["step"], valid[col], linewidth=2)
            ax.axhline(0, color="gray", linestyle="--", alpha=0.5)
            ax.set_title(col); ax.set_xlabel("Step"); ax.set_ylabel("Reward")
            ax.grid(True, alpha=0.3)
        plt.suptitle("GRPO Training Rewards Over Time", fontsize=13, fontweight="bold")
        plt.tight_layout()
        plt.savefig("training_rewards.png", dpi=150, bbox_inches="tight")
        plt.show()
        print("Plot saved to training_rewards.png")

## Phase 6: View the Results

In [ ]:
# Cell 22 — Save LoRA adapter
ADAPTER_PATH = "./lora_adapter"
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)

import os
print(f"Adapter saved to: {ADAPTER_PATH}")
for f in sorted(os.listdir(ADAPTER_PATH)):
    size = os.path.getsize(os.path.join(ADAPTER_PATH, f))
    print(f"  {f:<45} {size/1e6:.2f} MB")

In [ ]:
# Cell 23 — Define compare_old_and_new_model
# No changes needed in this cell.

def compare_old_and_new_model(messages):
    """
    Compare OLD model (LoRA disabled) vs NEW model (LoRA enabled) side by side.
    Accepts messages in chat format: list of {role, content} dicts.
    """
    text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    gen_kw = dict(max_new_tokens=512, temperature=0.1, do_sample=True,
                  pad_token_id=tokenizer.eos_token_id)

    FastLanguageModel.for_inference(model)

    # OLD: base model without LoRA adapters
    model.disable_adapters()
    with torch.no_grad():
        out_old = model.generate(**inputs, **gen_kw)
    old_resp = tokenizer.decode(out_old[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    # NEW: fine-tuned model with LoRA adapters
    model.enable_adapters()
    with torch.no_grad():
        out_new = model.generate(**inputs, **gen_kw)
    new_resp = tokenizer.decode(out_new[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    print("=" * 70)
    user_msg = next((m["content"] for m in messages if m["role"] == "user"), "")
    print(f"QUESTION: {user_msg}")
    print("=" * 70)
    print("\n── OLD MODEL (base, LoRA disabled) ──────────────────────────")
    print(old_resp)
    print("\n── NEW MODEL (fine-tuned, LoRA enabled) ─────────────────────")
    print(new_resp)
    print("=" * 70)
    return old_resp, new_resp

print("compare_old_and_new_model() ready.")

In [ ]:
# Cell 24 — TODO: Compare OLD vs NEW on a dataset example
# Load the first item from the dataset and compare both models.

messages = ds[0]["prompt"]
print(f"Testing on: word='{ds[0]['word']}', letter='{ds[0]['letter']}', answer='{ds[0]['answer']}'")
compare_old_and_new_model(messages)

In [ ]:
# Cell 25 — TODO: Check for catastrophic forgetting
# Ask both models a well-known general knowledge question.
# Both OLD and NEW should answer correctly — proving fine-tuning
# taught a new skill WITHOUT erasing existing knowledge.

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user",   "content": "What is the capital of Philippines?"},
]
compare_old_and_new_model(messages)

## Summary

| Component | Choice | Rationale |
|-----------|--------|-----------|
| `lora_rank` | 64 | Balanced expressiveness vs. 16 GB VRAM budget |
| `target_modules` | All 7 attention + MLP layers | Maximum adapter coverage for learning procedural reasoning |
| `learning_rate` | 1e-5 | Standard LoRA LR; avoids catastrophic forgetting |
| `beta` | 0.0001 | Small KL penalty → fast skill acquisition |
| `num_generations` | 4 | Sufficient group diversity for GRPO comparisons |

### Reward Function Summary

| Function | Correct Signal | Max | Min |
|----------|---------------|-----|-----|
| `format_reward_func` | XML format + digit answer | +1.0 | 0.0 |
| `numbering_reward_func` | Sequential step numbers | +0.5/step | −1.0/step |
| `spelling_reward_func` | Letter-by-letter spelling | +2.0 | variable |
| `counting_reward_func` | Accurate running total | +1.0 | −1.0 |
| `correct_answer_reward_func` | Final answer correct | +2.0 | −1.0 |